In [ ]:
# Cell 0 — Install Dependencies
!pip install minigrid stable-baselines3[extra] gymnasium matplotlib pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.7/136.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 7.8 MB/s eta 0:00:00


In [ ]:
# Cell 1 — Version Check
import minigrid
import stable_baselines3
import gymnasium
import matplotlib
import pandas

print(f"minigrid:          {minigrid.__version__}")
print(f"stable-baselines3: {stable_baselines3.__version__}")
print(f"gymnasium:         {gymnasium.__version__}")
print(f"matplotlib:        {matplotlib.__version__}")
print(f"pandas:            {pandas.__version__}")

import torch
print(f"torch:             {torch.__version__}")
print(f"CUDA available:    {torch.cuda.is_available()}")

minigrid:          3.0.0
stable-baselines3: 2.7.1
gymnasium:         1.2.3
matplotlib:        3.10.0
pandas:            2.2.2
torch:             2.10.0+cpu
CUDA available:    False


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Cell 2 — config.py
import os

# ─── Timesteps (override via env vars to scale up) ───────────────────────────
TIMESTEPS = {
    "easy":     int(os.getenv("EASY_STEPS",     150_000)),
    "medium":   int(os.getenv("MEDIUM_STEPS",   150_000)),
    "hard":     int(os.getenv("HARD_STEPS",     200_000)),
    "direct":   int(os.getenv("DIRECT_STEPS",   500_000)),
}

# ─── PPO Hyperparameters (identical for both experiments) ────────────────────
PPO_PARAMS = {
    "policy":        "MlpPolicy",
    "learning_rate": 3e-4,
    "n_steps":       512,
    "batch_size":    64,
    "n_epochs":      10,
    "gamma":         0.99,
    "ent_coef":      0.01,
    "verbose":       0,
}

# ─── Evaluation ──────────────────────────────────────────────────────────────
EVAL_EPISODES = 100

# ─── Paths ───────────────────────────────────────────────────────────────────
BASE_DIR     = "/content/curriculum_rl"
RESULTS_DIR  = f"{BASE_DIR}/results"
LOGS_DIR     = f"{RESULTS_DIR}/logs"
PLOTS_DIR    = f"{RESULTS_DIR}/plots"
MODELS_DIR   = f"{RESULTS_DIR}/models"

# ─── Create directories ──────────────────────────────────────────────────────
for path in [LOGS_DIR, PLOTS_DIR, MODELS_DIR]:
    os.makedirs(path, exist_ok=True)

# ─── Sanity print ────────────────────────────────────────────────────────────
print("✅ Config loaded")
print(f"   Timesteps  : {TIMESTEPS}")
print(f"   Eval eps   : {EVAL_EPISODES}")
print(f"   Base dir   : {BASE_DIR}")
print(f"   Directories: logs/ plots/ models/ — created ✅")

✅ Config loaded
   Timesteps  : {'easy': 150000, 'medium': 150000, 'hard': 200000, 'direct': 500000}
   Eval eps   : 100
   Base dir   : /content/curriculum_rl
   Directories: logs/ plots/ models/ — created ✅


In [ ]:
# Cell 3 — custom_env.py
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from minigrid.core.grid import Grid
from minigrid.core.mission import MissionSpace
from minigrid.core.world_object import Goal, Key, Door, Wall
from minigrid.minigrid_env import MiniGridEnv

# ─────────────────────────────────────────────────────────────
# BASE CLASS — shared logic for all environments
# ─────────────────────────────────────────────────────────────
class BaseKeyDoorEnv(MiniGridEnv):
    def __init__(self, size, layout_fn, max_steps=None, **kwargs):
        self.layout_fn = layout_fn
        mission_space = MissionSpace(mission_func=lambda: "pick up the key and open the door to reach the goal")
        if max_steps is None:
            max_steps = 4 * size * size
        super().__init__(
            mission_space=mission_space,
            grid_size=size,
            max_steps=max_steps,
            **kwargs
        )

    def _gen_grid(self, width, height):
        self.grid = Grid(width, height)
        # Outer walls
        self.grid.wall_rect(0, 0, width, height)
        # Call layout function to place objects
        self.layout_fn(self)

    def step(self, action):
        obs, reward, terminated, truncated, info = super().step(action)
        # Success = reached goal (terminated with positive reward)
        info["success"] = terminated and reward > 0
        return obs, reward, terminated, truncated, info


# ─────────────────────────────────────────────────────────────
# LAYOUT FUNCTIONS — fixed positions per environment
# ─────────────────────────────────────────────────────────────

def layout_easy(env):
    """6x6: Agent navigates to goal. No key, no door, no obstacles."""
    env.agent_pos = np.array([1, 1])
    env.agent_dir = 0
    env.put_obj(Goal(), 4, 4)

def layout_medium(env):
    """8x8: Agent picks up key, opens door, reaches goal."""
    env.agent_pos = np.array([1, 1])
    env.agent_dir = 0
    # Key
    env.put_obj(Key("yellow"), 2, 3)
    # Wall dividing grid
    for i in range(1, 7):
        env.put_obj(Wall(), 4, i)
    # Door in the wall
    env.put_obj(Door("yellow", is_locked=True), 4, 4)
    # Goal on the other side
    env.put_obj(Goal(), 6, 6)

def layout_hard(env):
    """10x10: Key + locked door + obstacles + goal."""
    env.agent_pos = np.array([1, 1])
    env.agent_dir = 0
    # Key
    env.put_obj(Key("yellow"), 2, 4)
    # Dividing wall
    for i in range(1, 9):
        env.put_obj(Wall(), 5, i)
    # Door
    env.put_obj(Door("yellow", is_locked=True), 5, 5)
    # Obstacles (walls scattered)
    for pos in [(2, 2), (3, 6), (4, 3), (1, 7)]:
        env.put_obj(Wall(), pos[0], pos[1])
    # Goal
    env.put_obj(Goal(), 8, 8)

def layout_test(env):
    """10x10: Different layout — never seen during training."""
    env.agent_pos = np.array([1, 8])   # agent starts bottom-left (different!)
    env.agent_dir = 0
    # Key in different position
    env.put_obj(Key("yellow"), 3, 6)
    # Dividing wall — horizontal this time (different structure!)
    for i in range(1, 9):
        env.put_obj(Wall(), i, 4)
    # Door in different position
    env.put_obj(Door("yellow", is_locked=True), 5, 4)
    # Different obstacle pattern
    for pos in [(2, 7), (6, 6), (7, 2), (3, 2)]:
        env.put_obj(Wall(), pos[0], pos[1])
    # Goal at top-right (different!)
    env.put_obj(Goal(), 8, 1)


# ─────────────────────────────────────────────────────────────
# REGISTERED ENVIRONMENTS
# ─────────────────────────────────────────────────────────────

class KeyDoorEasyEnv(BaseKeyDoorEnv):
    def __init__(self, **kwargs):
        super().__init__(size=6, layout_fn=layout_easy, **kwargs)

class KeyDoorMediumEnv(BaseKeyDoorEnv):
    def __init__(self, **kwargs):
        super().__init__(size=8, layout_fn=layout_medium, **kwargs)

class KeyDoorHardEnv(BaseKeyDoorEnv):
    def __init__(self, **kwargs):
        super().__init__(size=10, layout_fn=layout_hard, **kwargs)

class KeyDoorTestEnv(BaseKeyDoorEnv):
    def __init__(self, **kwargs):
        super().__init__(size=10, layout_fn=layout_test, **kwargs)


# ─────────────────────────────────────────────────────────────
# GYMNASIUM REGISTRATION
# ─────────────────────────────────────────────────────────────

def register_envs():
    env_list = [
        ("KeyDoorEasy-v0",   KeyDoorEasyEnv),
        ("KeyDoorMedium-v0", KeyDoorMediumEnv),
        ("KeyDoorHard-v0",   KeyDoorHardEnv),
        ("KeyDoorTest-v0",   KeyDoorTestEnv),
    ]
    for env_id, env_class in env_list:
        if env_id not in gym.envs.registry:
            gym.register(id=env_id, entry_point=lambda cls=env_class: cls())

register_envs()
print("✅ All 4 environments registered:")
print("   - KeyDoorEasy-v0   (6x6,  navigation only)")
print("   - KeyDoorMedium-v0 (8x8,  key + door)")
print("   - KeyDoorHard-v0   (10x10, key + door + obstacles) ← training")
print("   - KeyDoorTest-v0   (10x10, different layout)       ← test only")

✅ All 4 environments registered:
   - KeyDoorEasy-v0   (6x6,  navigation only)
   - KeyDoorMedium-v0 (8x8,  key + door)
   - KeyDoorHard-v0   (10x10, key + door + obstacles) ← training
   - KeyDoorTest-v0   (10x10, different layout)       ← test only


In [ ]:
# Cell 4 — Environment Sanity Check (fixed)
import gymnasium as gym
import numpy as np

def check_env(env_id, steps=20):
    print(f"\n{'─'*50}")
    print(f"Testing: {env_id}")
    env = gym.make(env_id)
    obs, info = env.reset()
    print(f"  Obs shape     : {obs['image'].shape}")
    print(f"  Action space  : {env.action_space}")
    print(f"  Max steps     : {env.unwrapped.max_steps}")  # ← fix here

    total_reward = 0
    for i in range(steps):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        if terminated or truncated:
            print(f"  Episode ended : step {i+1} | reward={reward:.3f} | success={info.get('success', False)}")
            obs, info = env.reset()

    print(f"  Total reward ({steps} steps): {total_reward:.3f}")
    print(f"  ✅ {env_id} OK")
    env.close()

for env_id in [
    "KeyDoorEasy-v0",
    "KeyDoorMedium-v0",
    "KeyDoorHard-v0",
    "KeyDoorTest-v0",
]:
    check_env(env_id)

print(f"\n{'─'*50}")
print("✅ All environments passed sanity check")


──────────────────────────────────────────────────
Testing: KeyDoorEasy-v0
  Obs shape     : (7, 7, 3)
  Action space  : Discrete(7)
  Max steps     : 144
  Total reward (20 steps): 0.000
  ✅ KeyDoorEasy-v0 OK

──────────────────────────────────────────────────
Testing: KeyDoorMedium-v0
  Obs shape     : (7, 7, 3)
  Action space  : Discrete(7)
  Max steps     : 256
  Total reward (20 steps): 0.000
  ✅ KeyDoorMedium-v0 OK

──────────────────────────────────────────────────
Testing: KeyDoorHard-v0
  Obs shape     : (7, 7, 3)
  Action space  : Discrete(7)
  Max steps     : 400
  Total reward (20 steps): 0.000
  ✅ KeyDoorHard-v0 OK

──────────────────────────────────────────────────
Testing: KeyDoorTest-v0
  Obs shape     : (7, 7, 3)
  Action space  : Discrete(7)
  Max steps     : 400
  Total reward (20 steps): 0.000
  ✅ KeyDoorTest-v0 OK

──────────────────────────────────────────────────
✅ All environments passed sanity check


In [ ]:
# Cell 5 — logger.py
import csv
import os
import numpy as np
from stable_baselines3.common.callbacks import BaseCallback

class TrainingLogger(BaseCallback):
    """
    Logs episode reward and success rate to a CSV file during training.
    Attaches as a SB3 callback — zero changes needed in training scripts.
    """

    def __init__(self, log_path: str, verbose=0):
        super().__init__(verbose)
        self.log_path    = log_path
        self.episode_rewards  = []
        self.episode_successes = []
        self._current_rewards  = {}   # track per-env episode reward

        # Create CSV with headers
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        with open(log_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["timestep", "episode", "reward", "success", "success_rate_100"])

        self._episode_count = 0
        print(f"   📝 Logger ready → {log_path}")

    def _on_step(self) -> bool:
        # SB3 vectorized env — infos and dones are lists
        for i, done in enumerate(self.locals["dones"]):
            if done:
                info    = self.locals["infos"][i]
                reward  = info.get("episode", {}).get("r", 0.0)
                success = int(info.get("success", False))

                self.episode_rewards.append(reward)
                self.episode_successes.append(success)
                self._episode_count += 1

                # Rolling success rate over last 100 episodes
                recent = self.episode_successes[-100:]
                success_rate = np.mean(recent) * 100

                with open(self.log_path, "a", newline="") as f:
                    writer = csv.writer(f)
                    writer.writerow([
                        self.num_timesteps,
                        self._episode_count,
                        round(reward, 4),
                        success,
                        round(success_rate, 2),
                    ])

        return True  # returning False would stop training


def load_log(log_path: str):
    """Load a training log CSV into a list of dicts."""
    import pandas as pd
    if not os.path.exists(log_path):
        raise FileNotFoundError(f"Log not found: {log_path}")
    df = pd.read_csv(log_path)
    print(f"   📂 Loaded log: {log_path} ({len(df)} episodes)")
    return df


# ─── Sanity check ────────────────────────────────────────────
print("✅ Logger defined")
print("   TrainingLogger  — SB3 callback, writes CSV per episode")
print("   load_log()      — reads CSV back as pandas DataFrame")

✅ Logger defined
   TrainingLogger  — SB3 callback, writes CSV per episode
   load_log()      — reads CSV back as pandas DataFrame


In [ ]:
# Cell 6+7+8 COMBINED — train both agents + evaluate in same cell
# No save/load issue — models stay in memory throughout
import os
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from minigrid.wrappers import FlatObsWrapper
from minigrid.core.grid import Grid
from minigrid.core.mission import MissionSpace
from minigrid.core.world_object import Goal, Key, Door, Wall
from minigrid.minigrid_env import MiniGridEnv
import pandas as pd
import csv

# ════════════════════════════════════════════════════════
# ENV DEFINITIONS
# ════════════════════════════════════════════════════════
class BaseKeyDoorEnv(MiniGridEnv):
    def __init__(self, size, layout_fn, max_steps=None, **kwargs):
        self.layout_fn = layout_fn
        mission_space  = MissionSpace(mission_func=lambda: "pick up the key and open the door to reach the goal")
        if max_steps is None:
            max_steps = 4 * size * size
        super().__init__(mission_space=mission_space, grid_size=size, max_steps=max_steps, **kwargs)

    def _gen_grid(self, width, height):
        self.grid = Grid(width, height)
        self.grid.wall_rect(0, 0, width, height)
        self.layout_fn(self)

    def step(self, action):
        obs, reward, terminated, truncated, info = super().step(action)
        info["success"] = terminated and reward > 0
        return obs, reward, terminated, truncated, info

def layout_easy(env):
    env.agent_pos = np.array([1, 1])
    env.agent_dir = 0
    env.put_obj(Goal(), 4, 4)

def layout_medium(env):
    env.agent_pos = np.array([1, 1])
    env.agent_dir = 0
    env.put_obj(Key("yellow"), 2, 3)
    for i in range(1, 7):
        env.put_obj(Wall(), 4, i)
    env.put_obj(Door("yellow", is_locked=True), 4, 4)
    env.put_obj(Goal(), 6, 6)

def layout_hard(env):
    env.agent_pos = np.array([1, 1])
    env.agent_dir = 0
    env.put_obj(Key("yellow"), 2, 4)
    for i in range(1, 9):
        env.put_obj(Wall(), 5, i)
    env.put_obj(Door("yellow", is_locked=True), 5, 5)
    for pos in [(2, 2), (3, 6), (4, 3), (1, 7)]:
        env.put_obj(Wall(), pos[0], pos[1])
    env.put_obj(Goal(), 8, 8)

def layout_test(env):
    env.agent_pos = np.array([1, 8])
    env.agent_dir = 0
    env.put_obj(Key("yellow"), 3, 6)
    for i in range(1, 9):
        env.put_obj(Wall(), i, 4)
    env.put_obj(Door("yellow", is_locked=True), 5, 4)
    for pos in [(2, 7), (6, 6), (7, 2), (3, 2)]:
        env.put_obj(Wall(), pos[0], pos[1])
    env.put_obj(Goal(), 8, 1)

class KeyDoorEasyEnv(BaseKeyDoorEnv):
    def __init__(self, **kwargs):
        super().__init__(size=6,  layout_fn=layout_easy,   **kwargs)

class KeyDoorMediumEnv(BaseKeyDoorEnv):
    def __init__(self, **kwargs):
        super().__init__(size=8,  layout_fn=layout_medium, **kwargs)

class KeyDoorHardEnv(BaseKeyDoorEnv):
    def __init__(self, **kwargs):
        super().__init__(size=10, layout_fn=layout_hard,   **kwargs)

class KeyDoorTestEnv(BaseKeyDoorEnv):
    def __init__(self, **kwargs):
        super().__init__(size=10, layout_fn=layout_test,   **kwargs)

def register_envs():
    for env_id, cls in [
        ("KeyDoorEasy-v0",   KeyDoorEasyEnv),
        ("KeyDoorMedium-v0", KeyDoorMediumEnv),
        ("KeyDoorHard-v0",   KeyDoorHardEnv),
        ("KeyDoorTest-v0",   KeyDoorTestEnv),
    ]:
        if env_id not in gym.envs.registry:
            gym.register(id=env_id, entry_point=lambda cls=cls: cls())

register_envs()
print("✅ Environments registered")

# ════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════
TIMESTEPS = {
    "easy":   int(os.getenv("EASY_STEPS",   150_000)),
    "medium": int(os.getenv("MEDIUM_STEPS", 150_000)),
    "hard":   int(os.getenv("HARD_STEPS",   200_000)),
    "direct": int(os.getenv("DIRECT_STEPS", 500_000)),
}
PPO_PARAMS = {
    "learning_rate": 3e-4,
    "n_steps":       512,
    "batch_size":    64,
    "n_epochs":      10,
    "gamma":         0.99,
    "ent_coef":      0.01,
    "verbose":       0,
}
LOGS_DIR   = "/content/curriculum_rl/results/logs"
MODELS_DIR = "/content/curriculum_rl/results/models"
os.makedirs(LOGS_DIR,   exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# ════════════════════════════════════════════════════════
# LOGGER
# ════════════════════════════════════════════════════════
from stable_baselines3.common.callbacks import BaseCallback

class TrainingLogger(BaseCallback):
    def __init__(self, log_path, verbose=0):
        super().__init__(verbose)
        self.log_path          = log_path
        self.episode_successes = []
        self.episode_rewards   = []
        self._episode_count    = 0
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        with open(log_path, "w", newline="") as f:
            csv.writer(f).writerow(["timestep","episode","reward","success","success_rate_100"])

    def _on_step(self):
        for i, done in enumerate(self.locals["dones"]):
            if done:
                info    = self.locals["infos"][i]
                reward  = info.get("episode", {}).get("r", 0.0)
                success = int(info.get("success", False))
                self.episode_rewards.append(reward)
                self.episode_successes.append(success)
                self._episode_count += 1
                sr = np.mean(self.episode_successes[-100:]) * 100
                with open(self.log_path, "a", newline="") as f:
                    csv.writer(f).writerow([self.num_timesteps, self._episode_count, round(reward,4), success, round(sr,2)])
        return True

# ════════════════════════════════════════════════════════
# ENV FACTORY
# ════════════════════════════════════════════════════════
def make_flat_env(env_id):
    def _init():
        env = gym.make(env_id)
        env = FlatObsWrapper(env)
        return env
    return _init

# ════════════════════════════════════════════════════════
# EVALUATION FUNCTION — uses model directly, no save/load
# ════════════════════════════════════════════════════════
def evaluate_model(model, env_id, n_episodes=100, label=""):
    """Evaluate model directly in memory — no save/load."""
    vec_env   = DummyVecEnv([make_flat_env(env_id)])
    successes = []
    rewards   = []

    for ep in range(n_episodes):
        obs       = vec_env.reset()
        done      = False
        ep_reward = 0
        success   = False
        while not done:
            action, _              = model.predict(obs, deterministic=True)
            obs, reward, dones, infos = vec_env.step(action)
            ep_reward             += reward[0]
            done                   = dones[0]
            if infos[0].get("success", False):
                success = True
        successes.append(int(success))
        rewards.append(ep_reward)

    vec_env.close()
    sr = np.mean(successes) * 100
    mr = np.mean(rewards)
    print(f"  [{label}] {env_id}")
    print(f"    Success Rate : {sr:.1f}%")
    print(f"    Mean Reward  : {mr:.4f}\n")
    return sr, mr

# ════════════════════════════════════════════════════════
# EXPERIMENT A — DIRECT TRAINING
# ════════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("EXPERIMENT A — Direct Training")
print(f"  Timesteps : {TIMESTEPS['direct']:,}")
print("=" * 55)

direct_env    = DummyVecEnv([make_flat_env("KeyDoorHard-v0")] * 4)
direct_logger = TrainingLogger(f"{LOGS_DIR}/direct_training.csv")

direct_model  = PPO(policy="MlpPolicy", env=direct_env, **PPO_PARAMS)
direct_model.learn(
    total_timesteps=TIMESTEPS["direct"],
    callback=direct_logger,
    progress_bar=True,
)
direct_env.close()

direct_sr = np.mean(direct_logger.episode_successes[-100:]) * 100
print(f"\n✅ Direct training complete")
print(f"   Training success rate (last 100 eps): {direct_sr:.1f}%")

# ════════════════════════════════════════════════════════
# EXPERIMENT B — CURRICULUM TRAINING
# ════════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("EXPERIMENT B — Curriculum Training")
print("=" * 55)

# Stage 1 — Easy
print("\n📚 STAGE 1 — Easy")
env_easy      = DummyVecEnv([make_flat_env("KeyDoorEasy-v0")] * 4)
logger_easy   = TrainingLogger(f"{LOGS_DIR}/curriculum_easy.csv")
curriculum_model = PPO(policy="MlpPolicy", env=env_easy, **PPO_PARAMS)
curriculum_model.learn(total_timesteps=TIMESTEPS["easy"], callback=logger_easy, progress_bar=True)
env_easy.close()
sr1 = np.mean(logger_easy.episode_successes[-100:]) * 100
print(f"   ✅ Stage 1 complete | Success rate: {sr1:.1f}%")

# Stage 2 — Medium
print("\n📚 STAGE 2 — Medium")
env_medium    = DummyVecEnv([make_flat_env("KeyDoorMedium-v0")] * 4)
logger_medium = TrainingLogger(f"{LOGS_DIR}/curriculum_medium.csv")
curriculum_model.set_env(env_medium)
curriculum_model.learn(total_timesteps=TIMESTEPS["medium"], callback=logger_medium, progress_bar=True, reset_num_timesteps=False)
env_medium.close()
sr2 = np.mean(logger_medium.episode_successes[-100:]) * 100
print(f"   ✅ Stage 2 complete | Success rate: {sr2:.1f}%")

# Stage 3 — Hard
print("\n📚 STAGE 3 — Hard")
env_hard      = DummyVecEnv([make_flat_env("KeyDoorHard-v0")] * 4)
logger_hard   = TrainingLogger(f"{LOGS_DIR}/curriculum_hard.csv")
curriculum_model.set_env(env_hard)
curriculum_model.learn(total_timesteps=TIMESTEPS["hard"], callback=logger_hard, progress_bar=True, reset_num_timesteps=False)
env_hard.close()
sr3 = np.mean(logger_hard.episode_successes[-100:]) * 100
print(f"   ✅ Stage 3 complete | Success rate: {sr3:.1f}%")

print(f"\n✅ Curriculum training complete")
print(f"   Easy   : {sr1:.1f}%")
print(f"   Medium : {sr2:.1f}%")
print(f"   Hard   : {sr3:.1f}%")

# ════════════════════════════════════════════════════════
# EVALUATION — both models in memory, no save/load
# ════════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("EVALUATION — Training Environment (KeyDoorHard-v0)")
print("=" * 55)
direct_train_sr,     direct_train_rew     = evaluate_model(direct_model,     "KeyDoorHard-v0", 100, "Direct")
curriculum_train_sr, curriculum_train_rew = evaluate_model(curriculum_model, "KeyDoorHard-v0", 100, "Curriculum")

print("=" * 55)
print("EVALUATION — Test Environment (KeyDoorTest-v0)")
print("Never seen during training")
print("=" * 55)
direct_test_sr,     direct_test_rew     = evaluate_model(direct_model,     "KeyDoorTest-v0", 100, "Direct")
curriculum_test_sr, curriculum_test_rew = evaluate_model(curriculum_model, "KeyDoorTest-v0", 100, "Curriculum")

# ════════════════════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════════════════════
print("=" * 55)
print("FINAL RESULTS SUMMARY")
print("=" * 55)
print(f"{'Metric':<35} {'Direct':>8} {'Curriculum':>12}")
print("-" * 55)
print(f"{'Train Success Rate (KeyDoorHard)':<35} {direct_train_sr:>7.1f}% {curriculum_train_sr:>11.1f}%")
print(f"{'Test Success Rate  (KeyDoorTest)':<35} {direct_test_sr:>7.1f}% {curriculum_test_sr:>11.1f}%")
print(f"{'Train Mean Reward':<35} {direct_train_rew:>8.4f} {curriculum_train_rew:>12.4f}")
print(f"{'Test Mean Reward':<35} {direct_test_rew:>8.4f} {curriculum_test_rew:>12.4f}")
print("=" * 55)

# Save results
df = pd.DataFrame({
    "agent":         ["Direct",         "Curriculum"],
    "train_success": [direct_train_sr,  curriculum_train_sr],
    "test_success":  [direct_test_sr,   curriculum_test_sr],
    "train_reward":  [direct_train_rew, curriculum_train_rew],
    "test_reward":   [direct_test_rew,  curriculum_test_rew],
})
save_path = f"{LOGS_DIR}/evaluation_results.csv"
df.to_csv(save_path, index=False)
print(f"\n✅ Results saved → {save_path}")
print(df.to_string(index=False))

✅ Environments registered

EXPERIMENT A — Direct Training
  Timesteps : 500,000


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):


✅ Direct training complete
   Training success rate (last 100 eps): 100.0%

EXPERIMENT B — Curriculum Training

📚 STAGE 1 — Easy


   ✅ Stage 1 complete | Success rate: 100.0%

📚 STAGE 2 — Medium


   ✅ Stage 2 complete | Success rate: 0.0%

📚 STAGE 3 — Hard


   ✅ Stage 3 complete | Success rate: 2.0%

✅ Curriculum training complete
   Easy   : 100.0%
   Medium : 0.0%
   Hard   : 2.0%

EVALUATION — Training Environment (KeyDoorHard-v0)
  [Direct] KeyDoorHard-v0
    Success Rate : 100.0%
    Mean Reward  : 0.9438

  [Curriculum] KeyDoorHard-v0
    Success Rate : 0.0%
    Mean Reward  : 0.0000

EVALUATION — Test Environment (KeyDoorTest-v0)
Never seen during training
  [Direct] KeyDoorTest-v0
    Success Rate : 0.0%
    Mean Reward  : 0.0000

  [Curriculum] KeyDoorTest-v0
    Success Rate : 0.0%
    Mean Reward  : 0.0000

FINAL RESULTS SUMMARY
Metric                                Direct   Curriculum
-------------------------------------------------------
Train Success Rate (KeyDoorHard)      100.0%         0.0%
Test Success Rate  (KeyDoorTest)        0.0%         0.0%
Train Mean Reward                     0.9438       0.0000
Test Mean Reward                      0.0000       0.0000

✅ Results saved → /content/curriculum_rl/results/logs/evalu

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
